In [45]:
import pandas as pd
import folium
from folium.plugins import AntPath


In [46]:
test_data = {'攤位':['粒粒冰','地瓜球'],
             '緯度':[24.061,20.065],
             '經度':[120.691,120.695]}
df_test_data = pd.DataFrame(test_data)

for index, row in df_test_data.iterrows():
    print(f'序號{index} 攤位:{row['攤位']} 緯度:{row['緯度']} 經度:{row['經度']}')

序號0 攤位:粒粒冰 緯度:24.061 經度:120.691
序號1 攤位:地瓜球 緯度:20.065 經度:120.695


In [47]:
data = [
    {"from_sna": "住宅區A", "from_ll": [22.645, 120.325], "to_sna": "凹子底捷運站", "to_ll": [22.660, 120.304], "count": 50},
    {"from_sna": "住宅區B", "from_ll": [22.655, 120.315], "to_sna": "凹子底捷運站", "to_ll": [22.660, 120.304], "count": 30},
    {"from_sna": "住宅區C", "from_ll": [22.615, 120.300], "to_sna": "中央公園捷運站", "to_ll": [22.624, 120.302], "count": 80},
    {"from_sna": "住宅區D", "from_ll": [22.620, 120.285], "to_sna": "鹽埕埔捷運站", "to_ll": [22.623, 120.284], "count": 45},
]
data

[{'from_sna': '住宅區A',
  'from_ll': [22.645, 120.325],
  'to_sna': '凹子底捷運站',
  'to_ll': [22.66, 120.304],
  'count': 50},
 {'from_sna': '住宅區B',
  'from_ll': [22.655, 120.315],
  'to_sna': '凹子底捷運站',
  'to_ll': [22.66, 120.304],
  'count': 30},
 {'from_sna': '住宅區C',
  'from_ll': [22.615, 120.3],
  'to_sna': '中央公園捷運站',
  'to_ll': [22.624, 120.302],
  'count': 80},
 {'from_sna': '住宅區D',
  'from_ll': [22.62, 120.285],
  'to_sna': '鹽埕埔捷運站',
  'to_ll': [22.623, 120.284],
  'count': 45}]

In [48]:
df_flow = pd.DataFrame(data)
m = folium.Map(location=[22.63,120.30], zoom_start=13, tiles='cartodbdark_matter')
m.save('test.html')
df_flow

,from_sna,from_ll,to_sna,to_ll,count
0,住宅區A,"[22.645, 120.325]",凹子底捷運站,"[22.66, 120.304]",50
1,住宅區B,"[22.655, 120.315]",凹子底捷運站,"[22.66, 120.304]",30
2,住宅區C,"[22.615, 120.3]",中央公園捷運站,"[22.624, 120.302]",80
3,住宅區D,"[22.62, 120.285]",鹽埕埔捷運站,"[22.623, 120.284]",45


In [49]:
for index, row in df_flow.iterrows():
    print(f'index{index} count:{row['count']}')

index0 count:50
index1 count:30
index2 count:80
index3 count:45


## 流量地圖視覺化

使用 `folium` 與 `AntPath` 將起點到終點的流量資料繪製成互動式動態地圖。

### 流程概覽

| 步驟 | 說明 |
|------|------|
| 1 | 計算所有座標的平均值作為地圖中心 |
| 2 | 建立 Folium 地圖物件 |
| 3 | 定義 Min-Max 正規化函式，將 count 轉為線寬 |
| 4 | 畫 AntPath 動態流向線 |
| 5 | 標記起點（綠色）與終點（紅色）圓形圖標 |
| 6 | 嵌入自訂 HTML 圖例 |
| 7 | 儲存為 HTML 檔案 |

### 使用套件

- **`folium`**：基於 Leaflet.js 的 Python 互動地圖套件
- **`folium.plugins.AntPath`**：繪製帶有流動螞蟻動畫效果的路徑
- **`folium.CircleMarker`**：在地圖上繪製圓形標記點

In [50]:
# ══════════════════════════════════════════════════════════════
# 步驟 1：計算地圖中心點
# ══════════════════════════════════════════════════════════════
# 將所有起點與終點的緯度、經度分別收集成兩個清單
all_lats = [ll[0] for ll in df_flow['from_ll']] + [ll[0] for ll in df_flow['to_ll']]
all_lons = [ll[1] for ll in df_flow['from_ll']] + [ll[1] for ll in df_flow['to_ll']]

# 取平均值作為地圖初始中心，讓所有點都落在視野內
center = [sum(all_lats) / len(all_lats), sum(all_lons) / len(all_lons)]

# ══════════════════════════════════════════════════════════════
# 步驟 2：建立 Folium 地圖物件
# ══════════════════════════════════════════════════════════════
# location  : 地圖初始中心座標 [緯度, 經度]
# zoom_start: 初始縮放等級（數字越大越近）
# tiles     : 底圖樣式，'CartoDB positron' 為淺色簡潔風格
m = folium.Map(location=center, zoom_start=14, tiles='CartoDB positron')

# ══════════════════════════════════════════════════════════════
# 步驟 3：定義線寬正規化函式
# ══════════════════════════════════════════════════════════════
# 目的：將 count 數值對應到視覺線寬（min_w ~ max_w 之間）
# 公式：Min-Max 正規化，count 越大 → 線越粗
count_min, count_max = df_flow['count'].min(), df_flow['count'].max()

def scale_weight(count, min_w=3, max_w=12):
    # 若所有 count 相同則回傳中間值，避免除以零
    if count_max == count_min:
        return (min_w + max_w) / 2
    return min_w + (count - count_min) / (count_max - count_min) * (max_w - min_w)

# ══════════════════════════════════════════════════════════════
# 步驟 4：畫 AntPath 動態流向線
# ══════════════════════════════════════════════════════════════
# AntPath 是 Folium 的動畫外掛，可畫出帶有流動螞蟻效果的路徑
for _, row in df_flow.iterrows():
    weight = scale_weight(row['count'])   # 依 count 計算線寬
    AntPath(
        locations=[row['from_ll'], row['to_ll']],  # 起點 → 終點座標
        weight=weight,          # 線寬（由 count 決定）
        color='#3388ff',        # 主線顏色（藍色）
        opacity=0.8,            # 透明度（0~1）
        delay=800,              # 動畫速度（毫秒），數字越小流動越快
        dash_array=[10, 20],    # 虛線樣式：[線段長, 間距]
        pulse_color='#ffffff',  # 流動螢光點的顏色（白色）
        tooltip=f"{row['from_sna']} → {row['to_sna']}  count: {row['count']}"  # 滑鼠懸停提示
    ).add_to(m)

# ══════════════════════════════════════════════════════════════
# 步驟 5：在地圖上標記起點與終點圓形標記
# ══════════════════════════════════════════════════════════════
# 用 set 記錄已畫過的座標，避免同一地點重複疊加標記
drawn_from = set()
drawn_to   = set()

for _, row in df_flow.iterrows():

    # ── 起點（綠色圓點）──
    key_from = tuple(row['from_ll'])       # 將 list 轉 tuple 才能存入 set
    if key_from not in drawn_from:         # 尚未畫過才新增
        folium.CircleMarker(
            location=row['from_ll'],
            radius=8,                      # 圓的半徑（像素）
            color='green',                 # 外框顏色
            fill=True,                     # 是否填色
            fill_color='green',            # 填色顏色
            fill_opacity=0.85,             # 填色透明度
            tooltip=f"起點：{row['from_sna']}"
        ).add_to(m)
        drawn_from.add(key_from)

    # ── 終點（紅色圓點）──
    key_to = tuple(row['to_ll'])
    if key_to not in drawn_to:
        folium.CircleMarker(
            location=row['to_ll'],
            radius=8,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.85,
            tooltip=f"終點：{row['to_sna']}"
        ).add_to(m)
        drawn_to.add(key_to)

# ══════════════════════════════════════════════════════════════
# 步驟 6：新增圖例（Legend）
# ══════════════════════════════════════════════════════════════
# 使用原始 HTML 字串建立圖例方塊，透過 folium.Element 嵌入地圖頁面
# position:fixed 讓圖例固定在畫面左下角，不隨地圖捲動
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
            background:white; padding:10px 16px; border-radius:8px;
            box-shadow:2px 2px 6px rgba(0,0,0,0.3); font-size:13px;">
  <b>圖例</b><br>
  <span style="color:green;">●</span> 起點（住宅區）<br>
  <span style="color:red;">●</span> 終點（捷運站）<br>
  <span style="color:#3388ff;">──▶</span> 流量（線寬 ∝ count）
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# ══════════════════════════════════════════════════════════════
# 步驟 7：將地圖儲存為 HTML 檔案
# ══════════════════════════════════════════════════════════════
# Folium 地圖本質上是一個 HTML 頁面，儲存後可用瀏覽器直接開啟
output_path = 'flow_map.html'
m.save(output_path)
print(f"地圖已儲存至 {output_path}")

地圖已儲存至 flow_map.html
